<a href="https://colab.research.google.com/github/nagmanisha967-ops/my-projects/blob/main/ManishaNag_random_forest_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing necessary libraries for data processing, machine learning, and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc
)

* **Explanation**:This cell imports all the essential Python libraries required for data manipulation, building the Random Forest classifier model, and generating evaluation metrics and visualization plots

In [ ]:
# Try to load the downloaded CSV dataset from Kaggle
try:
    df = pd.read_csv('breast-cancer.csv')
    print("Successfully loaded dataset from 'breast-cancer.csv'")
except FileNotFoundError:
    print("'breast-cancer.csv' not found. Loading the default breast cancer dataset from scikit-learn...")
    # Fallback to load_breast_cancer from sklearn
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['diagnosis'] = pd.Series(data.target).map({1: 'B', 0: 'M'}) # B for Benign (1), M for Malignant (0)

print("\nDataset Overview:")
print(df.head())

* **Explanation:** This cell loads your uploaded Kaggle dataset (breast-cancer.csv) into a Pandas DataFrame.

In [ ]:
print("\n--- Phase I: Exploratory Data Analysis ---")

# Check for class imbalance in the diagnosis variable
print("\nClass Distribution:")
print(df['diagnosis'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x='diagnosis', data=df, palette='viridis')
plt.title('diagnosis Class Distribution')
plt.xlabel('diagnosis Class')
plt.ylabel('Count')
plt.show()

# Create a numeric representation of diagnosis to safely compute correlations
df['diagnosis_numeric'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Correlation heatmap for the top numeric features
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
top_corr_features = numeric_df.corr()['diagnosis_numeric'].abs().sort_values(ascending=False).head(10).index
sns.heatmap(numeric_df[top_corr_features].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Top Feature Correlation Heatmap')
plt.show()

# Clean up the temporary column
df.drop(columns=['diagnosis_numeric'], inplace=True)

Explanation: This cell fulfills Phase I of the EDA guidelines. It inspects the class distribution of your diagnosis variable to check for potential class imbalance using a count plot, and generates a correlation heatmap to analyze relationships between the most important numerical variables before building the model.

In [ ]:
print("\n--- Phase II: Preprocessing ---")

# 1. Handle missing/null values
print("Missing values count per column:")
print(df.isnull().sum().sum())
df = df.dropna()

# 2. Separate features (X) and target (y)
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# Note on Feature Scaling:
# Feature scaling is skipped because Random Forest uses tree-based splitting, which is invariant to feature scales.

# 3. Train-Test Split (80:20 ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

* **Explanation:** This preprocessing step checks for missing values, separates input features (X) from the target label (y), and splits the dataset into an 80:20 training and testing ratio.

In [ ]:
print("\n--- Phase III: Model Development ---")

# Initialize and fit the RandomForestClassifier
# Hyperparameters:
# - n_estimators=100: Number of trees to balance stability and performance.
# - max_depth=10: Prevents individual trees from overfitting to the training set.
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

* **Explanation:** Here, the RandomForestClassifier is initialized with justified hyperparameters (n_estimators=100 and max_depth=10) and trained on the training set.

In [ ]:
print("\n--- Evaluation Metrics ---")

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred, labels=['B', 'M'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted B', 'Predicted M'],
            yticklabels=['Actual B', 'Actual M'])
plt.title('Confusion Matrix Heatmap')
plt.show()

# 2. Precision, Recall, and F1-Score calculation
# Specifying pos_label='M' since target labels are categorical strings ('B' and 'M')
p_score = precision_score(y_test, y_pred, pos_label='M')
r_score = recall_score(y_test, y_pred, pos_label='M')
f1 = f1_score(y_test, y_pred, pos_label='M')

print(f"Precision (Malignant): {p_score:.4f}")
print(f"Recall (Malignant):    {r_score:.4f}")
print(f"F1-Score (Malignant):  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 3. ROC-AUC Curve
# Convert string target to binary sequence dynamically for the ROC curve calculation
y_test_binary = (y_test == 'M').astype(int)
fpr, tpr, thresholds = roc_curve(y_test_binary, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

# 4. Feature Importance Bar Chart
importances = rf_model.feature_importances_
feature_names = X.columns
feat_importances = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
feat_importances.plot(kind='barh', color='teal')
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.show()

* **Explanation:** This final phase computes and plots all required project evaluation metrics, including the confusion matrix heatmap, precision, recall, F1-score, ROC-AUC curve, and a feature importance bar chart.